# Phân loại lỗi bề mặt tấm thép — Huấn luyện trên Google Colab

Notebook này **clone repo từ GitHub** rồi chạy pipeline huấn luyện 5 mô hình
(SVM, k-NN, Random Forest, Extra Trees, Gradient Boosting) với chiến lược **One-vs-Rest**.

**Cách dùng:** `Runtime` → `Run all` (hoặc `Ctrl+F9`). Không cần upload gì, không cần mount Drive.

Thời gian chạy dự kiến: **1–2 phút** trên CPU (dataset chỉ 1.941 mẫu).

Repo: <https://github.com/Vehicle666/steel-plate-fault-classification>

## 1. Lấy code + dữ liệu từ GitHub

Repo là public nên Colab clone trực tiếp, dữ liệu gốc đã nằm trong `data/raw/`.

In [ ]:
REPO = "https://github.com/Vehicle666/steel-plate-fault-classification.git"
PROJECT = REPO.rsplit("/", 1)[-1].removesuffix(".git")

import os

if os.path.isdir(PROJECT):
    %cd $PROJECT
    !git pull --ff-only
else:
    !git clone $REPO
    %cd $PROJECT

!git log --oneline -3
!ls data/raw

## 2. Cài thư viện

Colab đã có sẵn numpy/pandas/scikit-learn/matplotlib/seaborn; bước này chỉ đảm bảo
đúng phiên bản tối thiểu trong `requirements.txt`.

In [ ]:
!pip install -q -r requirements.txt
!python -c "import sklearn, pandas, numpy; print('sklearn', sklearn.__version__)"

## 3. Huấn luyện + đánh giá

Lệnh dưới đây chạy `src/train.py`: chia train/test stratified 80/20,
chuẩn hoá bằng `StandardScaler`, huấn luyện 5 mô hình OvR, xuất báo cáo và confusion matrix.

Có thể chỉnh tham số ngay trong lệnh, ví dụ `--test-size 0.3 --random-state 7`.

In [ ]:
!python -m src.train

## 4. Xem kết quả

In [ ]:
import pandas as pd
from IPython.display import Image, display

results = pd.read_csv("reports/results.csv")
display(results.style.format({
    "accuracy": "{:.4f}", "balanced_accuracy": "{:.4f}",
    "f1_macro": "{:.4f}", "f1_weighted": "{:.4f}",
    "fit_time_s": "{:.3f}", "predict_time_s": "{:.3f}",
}).background_gradient(subset=["f1_macro"], cmap="Blues").hide(axis="index"))

display(Image("reports/figures/metrics_comparison.png"))
display(Image("reports/figures/confusion_matrices_all.png"))

In [ ]:
from pathlib import Path

for name in ["SVM", "k-NN", "Random Forest", "Extra Trees", "Gradient Boosting"]:
    safe = name.lower().replace(" ", "_").replace("-", "")
    print(f"===== {name} =====")
    print(Path(f"reports/classification_report_{safe}.txt").read_text(encoding="utf-8"))

display(Image("reports/figures/confusion_matrix_gradient_boosting.png"))

## 5. (Tuỳ chọn) Tải kết quả về máy

Đóng gói `reports/` + `models/` thành zip để tải về, tiện đưa vào báo cáo.

In [ ]:
!zip -rq ket_qua.zip reports models

try:
    from google.colab import files

    files.download("ket_qua.zip")
except ImportError:
    print("Không chạy trên Colab — file nằm ở ket_qua.zip")